In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# Hunt+2024 catalog

https://ucc.ar/_clusters/ngc1039/

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Hunt2024", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1, save=True)

In [ ]:
df = tabs[0].to_pandas()
df_mem = tabs[1].to_pandas()
df_mem.head()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({#"Cluster": "Name",
                        # "Source": "GaiaDR3",
                        # "Proba": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.head()

In [ ]:
from wakai.catalog import get_nexsci_data

table='pscomppars'
nexsci = get_nexsci_data(table, clobber=False)
nexsci.tail()

In [ ]:
nexsci.shape

In [ ]:
if table=='ps':
    nexsci_hosts = nexsci[nexsci['default_flag']==1]
else:
    nexsci_hosts = nexsci.copy()
nexsci_hosts.shape

In [ ]:
nexsci.hostname.unique().shape

# cross-match nexsci with catalog

In [ ]:
import pandas as pd

if True:
    #from astroquery.xmatch
    matches = pd.read_csv('../data/nexsci_GaiaDR3_xmatch.csv')
if False:
    #from vizier query/exofop; not yet available
    matches = pd.read_csv('../data/nexsci_with_Gaiaid.csv')
    matches['GaiaDR3'] = matches['GaiaDR3_vizier']
matches.head()

In [ ]:
final = pd.merge(df_mem, matches, on='GaiaDR3')
final.head()

In [ ]:
final[['pl_name','Name']]

In [ ]:
final.hostname.unique().shape

In [ ]:
# pscomppars has no default_flag (it is already one row per planet)
final_default = final[final['default_flag']==1] if 'default_flag' in final else final
final_default.shape

In [ ]:
final_default.sort_values(by='pl_name')\
    .to_csv(f'../data/{cd.catalog_name}/nexsci_in_{cd.catalog_name}.csv', 
            index=False)